In [1]:
from typing import Any

import torch
import torch.nn as nn
import numpy as np

In [2]:
# class DGMLayer_(nn.Module):
#     """
#     Georgias Detorakis (2024): Practical Aspects on Solving Differential Equations Using Deep Learning: A Primer

#     """

#     def __init__(self, input_dim=1, hidden_size=50):
#         super().__init__()

#         self.I_zu = nn.Linear(input_dim, hidden_size)
#         self.Z_wg = nn.Linear(hidden_size, hidden_size)
#         self.Z_ug = nn.Linear(input_dim, hidden_size, bias=False)

#         self.G_wz = nn.Linear(hidden_size, hidden_size)
#         self.G_uz = nn.Linear(input_dim, hidden_size, bias=False)

#         self.R_wr = nn.Linear(hidden_size, hidden_size)
#         self.R_ur = nn.Linear(input_dim, hidden_size, bias=False)

#         self.H_wh = nn.Linear(hidden_size, hidden_size)
#         self.H_uh = nn.Linear(input_dim, hidden_size, bias=False)

#         # Non−linear Activation function
#         self.sigma = nn.Tanh()

#     def forward(self, x, s):
#         I = self.I_zu(s)
#         print(f"I shape = {I.shape}")
#         Z = self.sigma(self.Z_wg(I) + self.Z_ug(x))
#         print(f"Z shape = {Z.shape}")
#         G = self.sigma(self.G_wz(Z) + self.G_uz(x))
#         print(f"G shape = {G.shape}")
#         R = self.sigma(self.R_wr(G) + self.R_ur(x))
#         print(f"R shape {R.shape} and s shape {s.shape} and I shape {self.H_wh(I).shape}")
#         H = self.sigma(self.H_wh(I) * R + self.H_uh(x))
#         print(f"H shape {H.shape}")
#         out = (1-G)*H + Z*self.H_wh(I)
#         print(f"out shape {out.shape}")
#         # out = torch.sub(1, G) * H + Z * s
#         return out



In [3]:
class DGMLayer(nn.Module):
    """
    Georgias Detorakis (2024): Practical Aspects on Solving Differential Equations Using Deep Learning: A Primer

    """

    def __init__(self, input_dim=1, hidden_size=50):
        super().__init__()

        self.Z_wg = nn.Linear(hidden_size, hidden_size)
        self.Z_ug = nn.Linear(input_dim, hidden_size, bias=False)

        self.G_wz = nn.Linear(hidden_size, hidden_size)
        self.G_uz = nn.Linear(input_dim, hidden_size, bias=False)

        self.R_wr = nn.Linear(hidden_size, hidden_size)
        self.R_ur = nn.Linear(input_dim, hidden_size, bias=False)

        self.H_wh = nn.Linear(hidden_size, hidden_size)
        self.H_uh = nn.Linear(input_dim, hidden_size, bias=False)

        # Non−linear Activation function
        self.sigma = nn.Tanh()

    def forward(self, x, s):
#         I = self.I_zu(s)
#         print(f"I shape = {I.shape}")
        Z = self.sigma(self.Z_wg(s) + self.Z_ug(x))
        print(f"Z shape = {Z.shape}")
        G = self.sigma(self.G_wz(Z) + self.G_uz(x))
        print(f"G shape = {G.shape}")
        R = self.sigma(self.R_wr(G) + self.R_ur(x))
        print(f"R shape {R.shape} and s shape {s.shape} and I shape {self.H_wh(s).shape}")
        H = self.sigma(self.H_wh(s) * R + self.H_uh(x))
        print(f"H shape {H.shape}")
        out = (1-G)*H + Z*self.H_wh(s)
        print(f"out shape {out.shape}")
        # out = torch.sub(1, G) * H + Z * s
        return out



In [4]:
class DGMLayer0(nn.Module):

    def __init__(self, input_dim=1, hidden_size=50):
        super().__init__()

        self.I_zu = nn.Linear(input_dim, hidden_size)
        self.dgm_layer = DGMLayer(input_dim, hidden_size)
    

    def forward(self, x, s):
        s1 = self.I_zu(s)
        print(f"s1 shape {s1.shape}")
        out = self.dgm_layer(x,s1)
        print(f"out shape {out.shape}")
        return out



In [5]:
class DGMLayerN(nn.Module):

    def __init__(self, input_dim=1, output_dim=1, hidden_size=50):
        super().__init__()

        self.dgm_layer = DGMLayer(input_dim, hidden_size)
        self.K_zu = nn.Linear(hidden_size, output_dim)
    

    def forward(self, x, s):
        
        x1 = self.dgm_layer(x,s)
        print(f"x1 shape {x1.shape}")
        out = self.K_zu(x1)
        print(f"out shape {out.shape}")
        return out

In [6]:
def build_input(q, t_norm, gpos):
    # q: (B,7), t_norm: (B,1), gpos: (B,3)
    return torch.cat([q, t_norm, gpos], dim=-1)

In [7]:
def sample_goals(n):
    # Sample from xyz boundaries (cuboid) from which
    # to train the NN to generate plans to reach points inside this goals cuboid/region
    xs = np.random.uniform(0.25, 0.65, (n, 1))
    ys = np.random.uniform(-0.30, 0.30, (n, 1))
    zs = np.random.uniform(0.10, 0.60, (n, 1))
    return np.hstack([xs, ys, zs]).astype(np.float64)

In [8]:
jmin = np.array([-2.8973, -1.7628, -2.8973, -3.0718, -2.8973, -0.0175, -2.8973], dtype=np.float64)
jmax = np.array([2.8973, 1.7628, 2.8973, -0.0698, 2.8973, 3.7525, 2.8973], dtype=np.float64)
batch = 192
T = 2
Qp = 10

In [9]:
q_np = np.random.uniform(jmin, jmax, (batch, 7)).astype(np.float64)
t_np = np.random.uniform(0.0, T, (batch, 1)).astype(np.float64)
g_np = sample_goals(batch)

In [10]:
g_np.shape

(192, 3)

In [11]:


# running cost via FK (position-only)
# l_np = position_loss_fn(fk, joint_names, batch, Qp, g_np, q_np)

l_np = np.zeros((batch,), dtype=np.float64)
for i in range(batch):
    try:
#         p = fk.ee_position(joint_names, q_np[i])  # fk client gets coordinate position of hand/end-effector
        p = np.random.rand(3) 
        e = p - g_np[i]  # distance between current joint position i and goal position i
        l_np[i] = Qp * float(np.dot(e, e))
    except Exception:
        rospy.logwarn("fk_pos l: couldn't retrieve fk position")
        l_np[i] = 1e3


In [12]:
device = torch.device('cpu')

In [13]:
l_np.shape

(192,)

In [14]:
q = torch.tensor(q_np, dtype=torch.float32, device=device, requires_grad=True)
t = torch.tensor((t_np / T), dtype=torch.float32, device=device, requires_grad=True)
g = torch.tensor(g_np, dtype=torch.float32, device=device)
l = torch.tensor(l_np, dtype=torch.float32, device=device)

In [15]:
print(q.shape)
print(t.shape)
print(g.shape)
print(l.shape)

torch.Size([192, 7])
torch.Size([192, 1])
torch.Size([192, 3])
torch.Size([192])


In [16]:
inp = build_input(q, t, g)

In [17]:
inp.shape

torch.Size([192, 11])

In [18]:
inp.T.shape

torch.Size([11, 192])

In [19]:
dgm_layer = DGMLayer(input_dim=11, hidden_size=192)

In [20]:
dgm_layer

DGMLayer(
  (Z_wg): Linear(in_features=192, out_features=192, bias=True)
  (Z_ug): Linear(in_features=11, out_features=192, bias=False)
  (G_wz): Linear(in_features=192, out_features=192, bias=True)
  (G_uz): Linear(in_features=11, out_features=192, bias=False)
  (R_wr): Linear(in_features=192, out_features=192, bias=True)
  (R_ur): Linear(in_features=11, out_features=192, bias=False)
  (H_wh): Linear(in_features=192, out_features=192, bias=True)
  (H_uh): Linear(in_features=11, out_features=192, bias=False)
  (sigma): Tanh()
)

In [21]:
dgm_layer_0 = DGMLayer0(input_dim=11, hidden_size=192)

In [22]:
dgm_layer_0

DGMLayer0(
  (I_zu): Linear(in_features=11, out_features=192, bias=True)
  (dgm_layer): DGMLayer(
    (Z_wg): Linear(in_features=192, out_features=192, bias=True)
    (Z_ug): Linear(in_features=11, out_features=192, bias=False)
    (G_wz): Linear(in_features=192, out_features=192, bias=True)
    (G_uz): Linear(in_features=11, out_features=192, bias=False)
    (R_wr): Linear(in_features=192, out_features=192, bias=True)
    (R_ur): Linear(in_features=11, out_features=192, bias=False)
    (H_wh): Linear(in_features=192, out_features=192, bias=True)
    (H_uh): Linear(in_features=11, out_features=192, bias=False)
    (sigma): Tanh()
  )
)

In [23]:
init = dgm_layer_0(inp,inp)

s1 shape torch.Size([192, 192])
Z shape = torch.Size([192, 192])
G shape = torch.Size([192, 192])
R shape torch.Size([192, 192]) and s shape torch.Size([192, 192]) and I shape torch.Size([192, 192])
H shape torch.Size([192, 192])
out shape torch.Size([192, 192])
out shape torch.Size([192, 192])


In [24]:
# V = model(build_input(q, t, g))
# loss_pde = hjb_residual_loss(V, q, t, l,
#                              R_inv_diag)  # hjb_residual_loss(V, q, t_norm, running_cost, R_inv_diag)



In [25]:
dgm_layer_n = DGMLayerN(input_dim=11,output_dim=192, hidden_size=192)

In [26]:
dgm_layer_n(inp, init)

Z shape = torch.Size([192, 192])
G shape = torch.Size([192, 192])
R shape torch.Size([192, 192]) and s shape torch.Size([192, 192]) and I shape torch.Size([192, 192])
H shape torch.Size([192, 192])
out shape torch.Size([192, 192])
x1 shape torch.Size([192, 192])
out shape torch.Size([192, 192])


tensor([[-0.6384, -0.1986, -0.0398,  ...,  1.1491,  0.0551, -0.4383],
        [ 0.4579,  0.2194,  0.6841,  ...,  0.4633,  0.1338,  0.3697],
        [-0.5513, -0.0485, -0.6695,  ...,  1.1108,  0.1429,  0.1832],
        ...,
        [ 0.3065, -0.3255, -0.0024,  ...,  0.5033,  0.4481, -0.5244],
        [ 0.7501,  1.1458,  0.2588,  ...,  0.2517, -0.1841,  0.2294],
        [ 0.2991,  0.1725,  0.2732,  ...,  0.6344, -0.1688,  0.0610]],
       grad_fn=<AddmmBackward0>)

In [27]:
class ValueNet(nn.Module):
    """
    num_dgm_layers 
    """

    def __init__(self, num_layers=1, input_dim=1, output_dim=1, hidden_size=50):
        super().__init__()
        
        self.layers = nn.ModuleList([DGMLayer0(input_dim, hidden_size)]) + \
        nn.ModuleList([DGMLayer(input_dim,hidden_size) for _ in range(num_layers)]) + \
                      nn.ModuleList([DGMLayerN(input_dim, output_dim, hidden_size)])   
            
#         self.dgm_layer = DGMLayer(input_dim, hidden_size)
    

    def forward(self, x, s):
        
        for i, layer in enumerate(self.layers):
            print(f"layer {i} = {layer}")
            x = layer(s,x)

        print(f"out shape: {x.shape}")
        return x.squeeze(-1)

In [28]:
v_net = ValueNet(num_layers=2, input_dim=11, output_dim=1, hidden_size=192)

In [29]:
v_net

ValueNet(
  (layers): ModuleList(
    (0): DGMLayer0(
      (I_zu): Linear(in_features=11, out_features=192, bias=True)
      (dgm_layer): DGMLayer(
        (Z_wg): Linear(in_features=192, out_features=192, bias=True)
        (Z_ug): Linear(in_features=11, out_features=192, bias=False)
        (G_wz): Linear(in_features=192, out_features=192, bias=True)
        (G_uz): Linear(in_features=11, out_features=192, bias=False)
        (R_wr): Linear(in_features=192, out_features=192, bias=True)
        (R_ur): Linear(in_features=11, out_features=192, bias=False)
        (H_wh): Linear(in_features=192, out_features=192, bias=True)
        (H_uh): Linear(in_features=11, out_features=192, bias=False)
        (sigma): Tanh()
      )
    )
    (1-2): 2 x DGMLayer(
      (Z_wg): Linear(in_features=192, out_features=192, bias=True)
      (Z_ug): Linear(in_features=11, out_features=192, bias=False)
      (G_wz): Linear(in_features=192, out_features=192, bias=True)
      (G_uz): Linear(in_features=11

In [30]:
v = v_net(inp,inp)

layer 0 = DGMLayer0(
  (I_zu): Linear(in_features=11, out_features=192, bias=True)
  (dgm_layer): DGMLayer(
    (Z_wg): Linear(in_features=192, out_features=192, bias=True)
    (Z_ug): Linear(in_features=11, out_features=192, bias=False)
    (G_wz): Linear(in_features=192, out_features=192, bias=True)
    (G_uz): Linear(in_features=11, out_features=192, bias=False)
    (R_wr): Linear(in_features=192, out_features=192, bias=True)
    (R_ur): Linear(in_features=11, out_features=192, bias=False)
    (H_wh): Linear(in_features=192, out_features=192, bias=True)
    (H_uh): Linear(in_features=11, out_features=192, bias=False)
    (sigma): Tanh()
  )
)
s1 shape torch.Size([192, 192])
Z shape = torch.Size([192, 192])
G shape = torch.Size([192, 192])
R shape torch.Size([192, 192]) and s shape torch.Size([192, 192]) and I shape torch.Size([192, 192])
H shape torch.Size([192, 192])
out shape torch.Size([192, 192])
out shape torch.Size([192, 192])
layer 1 = DGMLayer(
  (Z_wg): Linear(in_features=1

In [31]:
v.squeeze(-1).shape

torch.Size([192])

In [71]:
class DGMLayer_(nn.Module):
    """
    Georgias Detorakis (2024): Practical Aspects on Solving Differential Equations Using Deep Learning: A Primer

    """

    def __init__(self, input_dim=1, hidden_size=50, expansion_factor=2):
        super().__init__()
        
        self.expanded_hidden_size = expansion_factor*hidden_size

        self.Z_wg = nn.Linear(self.expanded_hidden_size, self.expanded_hidden_size)
        self.Z_ug = nn.Linear(input_dim, self.expanded_hidden_size, bias=False)

        self.G_wz = nn.Linear(expansion_factor*hidden_size, expansion_factor*hidden_size)
        self.G_uz = nn.Linear(input_dim, self.expanded_hidden_size, bias=False)

        self.R_wr = nn.Linear(self.expanded_hidden_size, self.expanded_hidden_size)
        self.R_ur = nn.Linear(input_dim, self.expanded_hidden_size, bias=False)

        self.H_wh = nn.Linear(self.expanded_hidden_size, self.expanded_hidden_size)
        self.H_uh = nn.Linear(input_dim, self.expanded_hidden_size, bias=False)

        # Non−linear Activation function
        self.sigma = nn.Tanh()

    def forward(self, x, s):
        
        Z = self.sigma(self.Z_wg(s) + self.Z_ug(x))
        print(f"Z shape = {Z.shape}")
        G = self.sigma(self.G_wz(Z) + self.G_uz(x))
        print(f"G shape = {G.shape}")
        R = self.sigma(self.R_wr(G) + self.R_ur(x))
        print(f"R shape {R.shape} and s shape {s.shape} and I shape {self.H_wh(s).shape}")
        H = self.sigma(self.H_wh(s) * R + self.H_uh(x))
        print(f"H shape {H.shape}")
        out = (1-G)*H + Z*self.H_wh(s)
        print(f"out shape {out.shape}")
        # out = torch.sub(1, G) * H + Z * s
        return out



In [72]:
class DGMLayer0_(nn.Module):

    def __init__(self, input_dim=1, hidden_size=50, expansion_factor=2):
        super().__init__()

        self.I_zu = nn.Linear(input_dim, expansion_factor*hidden_size)
        self.dgm_layer = DGMLayer_(input_dim, hidden_size, expansion_factor=2)
    

    def forward(self, x, s):
        s1 = self.I_zu(s)
        print(f"s1 shape {s1.shape}")
        out = self.dgm_layer(x,s1)
        print(f"out shape {out.shape}")
        return out

In [73]:
dgm_layer0_ = DGMLayer0_(input_dim=11, hidden_size=192)

In [74]:
dgm_layer0_

DGMLayer0_(
  (I_zu): Linear(in_features=11, out_features=384, bias=True)
  (dgm_layer): DGMLayer_(
    (Z_wg): Linear(in_features=384, out_features=384, bias=True)
    (Z_ug): Linear(in_features=11, out_features=384, bias=False)
    (G_wz): Linear(in_features=384, out_features=384, bias=True)
    (G_uz): Linear(in_features=11, out_features=384, bias=False)
    (R_wr): Linear(in_features=384, out_features=384, bias=True)
    (R_ur): Linear(in_features=11, out_features=384, bias=False)
    (H_wh): Linear(in_features=384, out_features=384, bias=True)
    (H_uh): Linear(in_features=11, out_features=384, bias=False)
    (sigma): Tanh()
  )
)

In [75]:
y = dgm_layer0_(inp,inp)
y.shape

s1 shape torch.Size([192, 384])
Z shape = torch.Size([192, 384])
G shape = torch.Size([192, 384])
R shape torch.Size([192, 384]) and s shape torch.Size([192, 384]) and I shape torch.Size([192, 384])
H shape torch.Size([192, 384])
out shape torch.Size([192, 384])
out shape torch.Size([192, 384])


torch.Size([192, 384])

In [76]:
dgm_layer_ = DGMLayer_(input_dim=11, hidden_size=192, expansion_factor=2)

In [82]:
z = dgm_layer_(inp, y)

Z shape = torch.Size([192, 384])
G shape = torch.Size([192, 384])
R shape torch.Size([192, 384]) and s shape torch.Size([192, 384]) and I shape torch.Size([192, 384])
H shape torch.Size([192, 384])
out shape torch.Size([192, 384])


In [92]:
class DGMLayerN_(nn.Module):

    def __init__(self, input_dim=1, output_dim=1, hidden_size=192, expansion_factor=2):
        super().__init__()
        
        self.expanded_hidden_size = expansion_factor*hidden_size

        self.dgm_layer = DGMLayer_(input_dim, hidden_size, expansion_factor)
        self.dgm_layerN_ = nn.Linear(expansion_factor*hidden_size, hidden_size)
        self.K_zu = nn.Linear(hidden_size, output_dim)
    

    def forward(self, x, s):
        
        x1 = self.dgm_layer(x,s)
        print(f"x1 shape {x1.shape}")
        x2 = self.dgm_layerN_(x1)
        print(f"x2 shape {x2.shape}")
        out = self.K_zu(x2)
        print(f"out shape {out.shape}")
        return out

In [93]:
dgm_layerN_ = DGMLayerN_(input_dim=11, output_dim=1, hidden_size=192, expansion_factor=2)

In [94]:
dgm_layerN_

DGMLayerN_(
  (dgm_layer): DGMLayer_(
    (Z_wg): Linear(in_features=384, out_features=384, bias=True)
    (Z_ug): Linear(in_features=11, out_features=384, bias=False)
    (G_wz): Linear(in_features=384, out_features=384, bias=True)
    (G_uz): Linear(in_features=11, out_features=384, bias=False)
    (R_wr): Linear(in_features=384, out_features=384, bias=True)
    (R_ur): Linear(in_features=11, out_features=384, bias=False)
    (H_wh): Linear(in_features=384, out_features=384, bias=True)
    (H_uh): Linear(in_features=11, out_features=384, bias=False)
    (sigma): Tanh()
  )
  (dgm_layerN_): Linear(in_features=384, out_features=192, bias=True)
  (K_zu): Linear(in_features=192, out_features=1, bias=True)
)

In [97]:
w = dgm_layerN_(inp, z)
w.shape

Z shape = torch.Size([192, 384])
G shape = torch.Size([192, 384])
R shape torch.Size([192, 384]) and s shape torch.Size([192, 384]) and I shape torch.Size([192, 384])
H shape torch.Size([192, 384])
out shape torch.Size([192, 384])
x1 shape torch.Size([192, 384])
x2 shape torch.Size([192, 192])
out shape torch.Size([192, 1])


torch.Size([192, 1])

In [99]:
class ValueNet_(nn.Module):
    """
    num_dgm_layers 
    """

    def __init__(self, num_layers=1, input_dim=1, output_dim=1, hidden_size=50, expansion_factor=2):
        super().__init__()
        
        self.layers = nn.ModuleList([DGMLayer0_(input_dim, hidden_size, expansion_factor)]) + \
        nn.ModuleList([DGMLayer_(input_dim, hidden_size, expansion_factor) for _ in range(num_layers)]) + \
                      nn.ModuleList([DGMLayerN_(input_dim, output_dim, hidden_size, expansion_factor)])   
            
#         self.dgm_layer = DGMLayer(input_dim, hidden_size)
    

    def forward(self, x, s):
        
        for i, layer in enumerate(self.layers):
            print(f"layer {i} = {layer}")
            x = layer(s,x)

        print(f"out shape: {x.shape}")
        return x.squeeze(-1)

In [100]:
v_net_ = ValueNet_(num_layers=2, input_dim=11, output_dim=1, hidden_size=192, expansion_factor=2)

In [101]:
v_net_

ValueNet_(
  (layers): ModuleList(
    (0): DGMLayer0_(
      (I_zu): Linear(in_features=11, out_features=384, bias=True)
      (dgm_layer): DGMLayer_(
        (Z_wg): Linear(in_features=384, out_features=384, bias=True)
        (Z_ug): Linear(in_features=11, out_features=384, bias=False)
        (G_wz): Linear(in_features=384, out_features=384, bias=True)
        (G_uz): Linear(in_features=11, out_features=384, bias=False)
        (R_wr): Linear(in_features=384, out_features=384, bias=True)
        (R_ur): Linear(in_features=11, out_features=384, bias=False)
        (H_wh): Linear(in_features=384, out_features=384, bias=True)
        (H_uh): Linear(in_features=11, out_features=384, bias=False)
        (sigma): Tanh()
      )
    )
    (1-2): 2 x DGMLayer_(
      (Z_wg): Linear(in_features=384, out_features=384, bias=True)
      (Z_ug): Linear(in_features=11, out_features=384, bias=False)
      (G_wz): Linear(in_features=384, out_features=384, bias=True)
      (G_uz): Linear(in_feature

In [103]:
v0 = v_net_(inp, inp)
v0.shape

layer 0 = DGMLayer0_(
  (I_zu): Linear(in_features=11, out_features=384, bias=True)
  (dgm_layer): DGMLayer_(
    (Z_wg): Linear(in_features=384, out_features=384, bias=True)
    (Z_ug): Linear(in_features=11, out_features=384, bias=False)
    (G_wz): Linear(in_features=384, out_features=384, bias=True)
    (G_uz): Linear(in_features=11, out_features=384, bias=False)
    (R_wr): Linear(in_features=384, out_features=384, bias=True)
    (R_ur): Linear(in_features=11, out_features=384, bias=False)
    (H_wh): Linear(in_features=384, out_features=384, bias=True)
    (H_uh): Linear(in_features=11, out_features=384, bias=False)
    (sigma): Tanh()
  )
)
s1 shape torch.Size([192, 384])
Z shape = torch.Size([192, 384])
G shape = torch.Size([192, 384])
R shape torch.Size([192, 384]) and s shape torch.Size([192, 384]) and I shape torch.Size([192, 384])
H shape torch.Size([192, 384])
out shape torch.Size([192, 384])
out shape torch.Size([192, 384])
layer 1 = DGMLayer_(
  (Z_wg): Linear(in_feature

torch.Size([192])